# 🎙️ ELEVEN AUDIO no Google Colab (Aceleração por GPU T4)
### Estúdio de Voz Local Completo — Clonagem de Voz, Síntese de Fala, Transcrição, Tradução e Dublagem

Este notebook instala e executa o **Eleven Audio** (desenvolvido por Gabriel Kendy) integrado ao motor neural **VoiceStudio / OmniVoice**, rodando 100% gratuito com aceleração por GPU no Google Colab.

- **Vídeo Tutorial Oficial:** [YouTube - Adeus ElevenLabs: Gerador e Clonador de Voz Profissional](https://www.youtube.com/watch?v=i41kJQpnCxM)
- **Repositório da Interface:** [github.com/gabrielkendy/eleven-audio](https://github.com/gabrielkendy/eleven-audio)
- **Repositório do Motor Neural:** [github.com/debpalash/VoiceStudio](https://github.com/debpalash/VoiceStudio)
- **Persistência Total no Google Drive:** Modelos (~3 GB), vozes clonadas e áudios gerados são salvos em `Meu Drive/ElevenAudio`, evitando downloads repetidos a cada sessão.
- **Acesso Web:** Link público gerado via Túnel Cloudflare + visualização integrada do Colab.

---
## 1. Verificação da GPU NVIDIA
> Certifique-se de que o ambiente está com GPU ativa: Menu superior **Ambiente de execução** ➔ **Alterar tipo de ambiente de execução** ➔ Selecione **T4 GPU** ➔ **Salvar**.

In [ ]:
import subprocess
import shutil

print("🔍 Verificando GPU do Google Colab...")
if shutil.which("nvidia-smi"):
    gpu_output = subprocess.check_output(["nvidia-smi"], text=True)
    print(gpu_output)
    print("\n✅ GPU NVIDIA detectada com sucesso! O estúdio funcionará em velocidade máxima.")
else:
    print("=" * 70)
    print("⚠️ ATENÇÃO: Nenhuma GPU NVIDIA detectada!")
    print("Para ativar:")
    print("1. Vá no menu superior: 'Ambiente de execução' -> 'Alterar tipo de ambiente de execução'")
    print("2. Selecione 'T4 GPU' e clique em 'Salvar'.")
    print("3. Execute esta célula novamente.")
    print("=" * 70)

---
## 2. Conectar Google Drive & Pastas de Persistência
> Conectar seu Google Drive garante que os modelos de IA, perfis de voz clonados e os áudios gerados nunca sejam perdidos quando a sessão do Colab encerrar.

In [ ]:
from google.colab import drive
import os

print("📂 Conectando ao Google Drive...")
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/ElevenAudio'
CACHE_HF = os.path.join(DRIVE_BASE, 'cache_hf')
CACHE_TORCH = os.path.join(DRIVE_BASE, 'cache_torch')
PASTA_SAIDAS = os.path.join(DRIVE_BASE, 'saidas')
PASTA_DADOS = os.path.join(DRIVE_BASE, 'dados')
PASTA_OMNIVOICE = os.path.join(DRIVE_BASE, 'omnivoice_data')

for pasta in [CACHE_HF, CACHE_TORCH, PASTA_SAIDAS, PASTA_DADOS, PASTA_OMNIVOICE]:
    os.makedirs(pasta, exist_ok=True)

# Configura variáveis de ambiente para persistência no Drive
os.environ['HF_HOME'] = CACHE_HF
os.environ['TRANSFORMERS_CACHE'] = CACHE_HF
os.environ['TORCH_HOME'] = CACHE_TORCH
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

# Variáveis do Eleven Audio Studio
os.environ['ESTUDIO_SAIDAS'] = PASTA_SAIDAS
os.environ['ESTUDIO_DADOS'] = PASTA_DADOS
os.environ['ESTUDIO_PORT'] = '7800'
os.environ['ESTUDIO_BASE_URL'] = 'http://127.0.0.1:3900'

# Variáveis do motor VoiceStudio
os.environ['OMNIVOICE_DATA_DIR'] = PASTA_OMNIVOICE
os.environ['OMNIVOICE_SERVER_MODE'] = '1'

print("✅ Google Drive configurado com sucesso!")
print(f"📁 Diretório de persistência: {DRIVE_BASE}")
print(f"   ├─ Modelos (Cache): {CACHE_HF}")
print(f"   ├─ Vozes e Perfis:  {PASTA_DADOS}")
print(f"   └─ Áudios Gerados:  {PASTA_SAIDAS}")

---
## 3. Instalação Completa do Eleven Audio & Motor Neural VoiceStudio
> Esta etapa baixa os repositórios oficiais, instala as dependências com o gerenciador ultrarrápido `uv`, aplica aceleração CUDA para áudio e configura os patches de resiliência.

In [ ]:
import os
import subprocess
import sys
import shutil

print("🚀 Iniciando instalação das dependências (primeira vez leva ~3 a 5 min)...")

# 1. Dependências de áudio do sistema operacional
print("📦 Instalando utilitários do sistema (ffmpeg, libsndfile)...")
!apt-get -qq update && apt-get -qq install -y ffmpeg libsndfile1

# 2. Gerenciador ultrarrápido uv
print("📦 Instalando gerenciador de pacotes uv...")
!pip install -q uv

%cd /content

# 3. Clonar VoiceStudio (base neural) se não existir
if not os.path.isdir('/content/VoiceStudio'):
    print("📥 Baixando motor neural VoiceStudio...")
    !git clone --depth 1 https://github.com/debpalash/VoiceStudio.git /content/VoiceStudio
    print("✅ VoiceStudio baixado!")
else:
    print("ℹ️ VoiceStudio já baixado.")

# 4. Clonar Eleven Audio (interface Gabriel Kendy) se não existir
if not os.path.isdir('/content/eleven-audio'):
    print("📥 Baixando interface Eleven Audio...")
    !git clone --depth 1 https://github.com/gabrielkendy/eleven-audio.git /content/eleven-audio
    print("✅ Eleven Audio baixado!")
else:
    print("ℹ️ Eleven Audio já baixado.")

# 5. Instalar dependências neurais do VoiceStudio com uv e restrições CUDA
print("📦 Instalando bibliotecas do motor neural (VoiceStudio)...\n(uv gerencia versões exatas de PyTorch e CUDA)")
constraints_flag = ["--constraint", "/content/VoiceStudio/deploy/torch-constraints.txt"] if os.path.exists("/content/VoiceStudio/deploy/torch-constraints.txt") else []
cmd_vs = ["uv", "pip", "install", "--system", "--no-cache"] + constraints_flag + ["/content/VoiceStudio"]
subprocess.run(cmd_vs, check=True)

# 6. Configurar cuDNN 8 para aceleração WhisperX
setup_script = "/content/VoiceStudio/scripts/setup.py"
if os.path.exists(setup_script):
    print("⚙️ Configurando aceleradores cuDNN para transcrição...")
    os.makedirs("/content/VoiceStudio/.venv", exist_ok=True)
    subprocess.run([sys.executable, setup_script], check=False)

# 7. Instalar dependências da interface Eleven Audio
print("📦 Instalando bibliotecas da interface Eleven Audio...")
!uv pip install --system -r /content/eleven-audio/requirements.txt
!uv pip install --system fastapi httpx uvicorn python-multipart pydub

# 8. Patches de resiliência e estabilidade
# Patch 1: Corrige web/gerar.js para tratamento robusto de respostas JSON/HTML
gerar_js_path = '/content/eleven-audio/web/gerar.js'
if os.path.exists(gerar_js_path):
    with open(gerar_js_path, 'r', encoding='utf-8') as f:
        js = f.read()
    old_api = 'const corpo = await resposta.json();\n      if (!resposta.ok) throw new Error(corpo.detail || "Não foi possível concluir.");'
    new_api = '''let corpo = null;
      const cType = resposta.headers.get("content-type") || "";
      if (cType.includes("application/json")) {
        try { corpo = await resposta.json(); } catch(_) {}
      }
      if (!resposta.ok) {
        if (corpo && corpo.detail) throw new Error(corpo.detail);
        if (resposta.status === 502 || resposta.status === 504) throw new Error("O motor recusou ou demorou para responder. Verifique se a voz de referência é limpa ou aguarde o modelo inicializar.");
        throw new Error(`Erro ${resposta.status}: Não foi possível concluir a operação.`);
      }'''
    if old_api in js:
        js = js.replace(old_api, new_api)
        with open(gerar_js_path, 'w', encoding='utf-8') as f:
            f.write(js)
        print("✅ Patch 1 aplicado: web/gerar.js protegido contra erros de rede.")

# Patch 2: Atualiza rotas.py para mensagens claras no front
rotas_py_path = '/content/eleven-audio/app/rotas.py'
if os.path.exists(rotas_py_path):
    with open(rotas_py_path, 'r', encoding='utf-8') as f:
        py = f.read()
    old_502 = 'raise HTTPException(502, f"erro da base: {erro}") from erro'
    new_422 = 'raise HTTPException(422, f"Aviso do motor: {erro}. Dica: Para OmniVoice, crie e selecione uma voz clonada na aba Clonar.") from erro'
    if old_502 in py:
        py = py.replace(old_502, new_422)
        with open(rotas_py_path, 'w', encoding='utf-8') as f:
            f.write(py)
        print("✅ Patch 2 aplicado: app/rotas.py com diagnósticos amigáveis.")

print("\n🎉 Instalação e configuração concluídas com sucesso!")

---
## 4. Pré-carregamento do Modelo Neural (OmniVoice)
> O motor **OmniVoice** é o clonador e sintetizador neural de alta fidelidade. Ao pré-baixar nesta célula, os arquivos ficam salvos permanentemente no seu Google Drive e as gerações de áudio serão praticamente instantâneas.

In [ ]:
from huggingface_hub import snapshot_download
import os

print("🔄 Pré-carregando pesos do modelo OmniVoice para o Google Drive...")
print(f"📁 Destino do cache HF: {os.environ.get('HF_HOME')}")

try:
    caminho_modelo = snapshot_download("k2-fsa/OmniVoice")
    print(f"\n✅ Modelo OmniVoice pronto e em cache: {caminho_modelo}")
except Exception as e:
    print(f"⚠️ Nota: Download em segundo plano ativo ({e}). O motor carregará automaticamente ao gerar.")

---
## 5. Iniciar os Serviços e Abrir o Estúdio de Voz
> Esta célula inicia o motor neural na porta 3900, a interface do estúdio na porta 7800 e gera o link seguro para você acessar do seu navegador ou celular.

In [ ]:
import subprocess
import time
import re
import os
import sys
import urllib.request
import json

# 1. Instalação do Cloudflare Tunnel
if not os.path.exists('/usr/local/bin/cloudflared'):
    print("🌐 Baixando binário Cloudflared...")
    !wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /usr/local/bin/cloudflared

# 2. Limpar qualquer processo prévio nas portas
subprocess.run("kill -9 $(lsof -t -i:3900) 2>/dev/null || true", shell=True)
subprocess.run("kill -9 $(lsof -t -i:7800) 2>/dev/null || true", shell=True)
time.sleep(1)

# 3. Iniciar motor neural VoiceStudio (porta 3900)
env_base = os.environ.copy()
env_base["OMNIVOICE_SERVER_MODE"] = "1"
env_base["OMNIVOICE_DATA_DIR"] = os.environ.get("OMNIVOICE_DATA_DIR", "/content/drive/MyDrive/ElevenAudio/omnivoice_data")
env_base["PYTHONUNBUFFERED"] = "1"

log_base_path = "/content/omnivoice_backend.log"
log_base = open(log_base_path, "ab")

print("🚀 1/2 Iniciando motor neural VoiceStudio (Porta 3900)...")
proc_base = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "main:app",
     "--app-dir", "backend", "--host", "127.0.0.1", "--port", "3900"],
    cwd="/content/VoiceStudio",
    env=env_base,
    stdout=log_base,
    stderr=subprocess.STDOUT
)

# Aguarda confirmação de saúde do motor
print("⏳ Aguardando motor neural inicializar...")
base_ativa = False
for _ in range(45):
    try:
        with urllib.request.urlopen("http://127.0.0.1:3900/health", timeout=3) as resp:
            if resp.status == 200:
                info = json.loads(resp.read().decode('utf-8'))
                device = info.get("device", "detectado")
                print(f"✅ Motor VoiceStudio ativo na porta 3900! [Dispositivo: {device}]")
                base_ativa = True
                break
    except Exception:
        pass
    if proc_base.poll() is not None:
        break
    time.sleep(2)

if not base_ativa:
    print("⚠️ O motor demorou a responder. Visualizando últimas linhas do log:")
    if os.path.exists(log_base_path):
        with open(log_base_path, "r", errors="ignore") as f:
            print("".join(f.readlines()[-25:]))

# 4. Iniciar Eleven Audio Studio (porta 7800)
env_app = os.environ.copy()
env_app["ESTUDIO_PORT"] = "7800"
env_app["ESTUDIO_BASE_URL"] = "http://127.0.0.1:3900"
env_app["ESTUDIO_SAIDAS"] = os.environ.get("ESTUDIO_SAIDAS", "/content/drive/MyDrive/ElevenAudio/saidas")
env_app["ESTUDIO_DADOS"] = os.environ.get("ESTUDIO_DADOS", "/content/drive/MyDrive/ElevenAudio/dados")
env_app["PYTHONUNBUFFERED"] = "1"

log_app_path = "/content/eleven_audio.log"
log_app = open(log_app_path, "ab")

print("🚀 2/2 Iniciando interface Eleven Audio Studio (Porta 7800)...")
proc_app = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.servidor:app",
     "--host", "127.0.0.1", "--port", "7800"],
    cwd="/content/eleven-audio",
    env=env_app,
    stdout=log_app,
    stderr=subprocess.STDOUT
)

app_ativa = False
for _ in range(25):
    try:
        with urllib.request.urlopen("http://127.0.0.1:7800", timeout=3) as resp:
            if resp.status == 200:
                print("✅ Interface Eleven Audio Studio ativa na porta 7800!")
                app_ativa = True
                break
    except Exception:
        pass
    time.sleep(1)

# 5. Abrir túnel Cloudflare seguro
print("🌐 Conectando túnel Cloudflare seguro...")
tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:7800', '--no-autoupdate'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for line in tunnel_proc.stdout:
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

print("\n" + "=" * 76)
if public_url:
    print("🎉 ELEVEN AUDIO ESTÚDIO NO AR COM SUCESSO!")
    print(f"👉 ACESSE SEU ESTÚDIO AQUI: {public_url}")
else:
    print("⚠️ Não foi possível obter o link do túnel automaticamente.")

try:
    from google.colab import output
    output.serve_kernel_port_as_window(7800)
    print("👉 Ou acesse pelo link embutido do Colab gerado acima!")
except Exception:
    pass

print("=" * 76)
print("""
🎙️ COMO USAR O ESTÚDIO (PASSO A PASSO DO TUTORIAL):

1. CLONAR UMA VOZ:
   - Abra a aba 'Clonar' na interface.
   - Envie um áudio limpo de 5 a 30 segundos (sem música de fundo).
   - Digite o nome da voz e a transcrição aproximada do que foi dito.
   - Marque o consentimento de autorização e clique em 'Clonar voz'.

2. GERAR FALA COM SUA VOZ CLONADA:
   - Abra a aba 'Compor' (Gerar).
   - Digite o texto que você deseja narrar.
   - No seletor de vozes, escolha a voz que você acabou de clonar.
   - Clique em 'Gerar áudio'. Em poucos segundos o áudio pronto tocará no player!

3. PERSISTÊNCIA TOTAL NO GOOGLE DRIVE:
   - Todos os áudios gerados ficam salvos automaticamente na pasta:
     'Meu Drive/ElevenAudio/saidas'
   - Seus perfis e modelos ficam preservados para a próxima vez!
""")
print("=" * 76 + "\n")
print("ℹ️ Mantenha esta célula em execução enquanto estiver usando o estúdio.")
print("   Para parar os serviços, clique no botão 'Parar' (Stop) da célula.")

try:
    while True:
        time.sleep(2)
except KeyboardInterrupt:
    print("\n🛑 Encerrando serviços...")
    proc_base.terminate()
    proc_app.terminate()
    tunnel_proc.terminate()
    print("✅ Todos os serviços foram desligados com sucesso.")